# dbt 1: Models and ref

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/dbt/dbt-01-models-and-ref.ipynb)

Run dbt locally against DuckDB using a fresh project for each lesson.

In [ ]:
# Setup: install notebook dependencies in Colab.
!pip install -q dbt-core~=1.9 dbt-duckdb~=1.9 duckdb~=2.0

## Setup and sample data

Create a small dbt project and database in a temporary directory.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import duckdb
from dbt.cli.main import dbtRunner

workspace = TemporaryDirectory()
project = Path(workspace.name)
models = project / "models"
models.mkdir()
warehouse = project / "warehouse.duckdb"
(project / "dbt_project.yml").write_text("name: handbook\nversion: '1.0'\nprofile: handbook\n")
(project / "profiles.yml").write_text(
    f"handbook:\n  outputs:\n    dev:\n      type: duckdb\n      path: '{warehouse}'\n      threads: 1\n"
    "  target: dev\n"
)


def dbt(*args, expect_success=True):
    result = dbtRunner().invoke([*args, "--project-dir", str(project), "--profiles-dir", str(project)])
    if result.success != expect_success:
        raise RuntimeError(f"Unexpected dbt result: {result.exception or result.result}")
    print("dbt completed:", result.success)


def query(sql):
    with duckdb.connect(str(warehouse)) as connection:
        print(connection.execute(sql).fetchall())

## Models and ref

A model is a SQL select that dbt builds into a relation. ref resolves another model and records a dependency.

**Where it matters**

Split cleaning and reporting into separately testable models.

**Watch out for**

Writing a literal relation name hides the dependency from dbt.

In [ ]:
(models / "stg_orders.sql").write_text("select * from (values (1, 1200), (2, 800)) t(id, cents)")
(models / "revenue.sql").write_text("select sum(cents) as total from {{ ref('stg_orders') }}")
dbt("build")
query("select total from revenue")

**What to notice:** dbt builds the staging model first. The output is 2000 cents.

## Check yourself

### Exercise 1: Apply the pattern

**Difficulty:** Easy · about 5 min

Which expression should a downstream model use to refer to revenue?

In [ ]:
# Try your solution here.

In [ ]:
print("{{ ref('revenue') }}")

## Key takeaways

- Split cleaning and reporting into separately testable models.

**Next:** [dbt 2](dbt-02-tests-and-contracts.ipynb)